# Stroke Prediction using TensorFlow/Keras Neural Network
This notebook applies exact preprocessing as silvano315 but replaces Random Forest with a highly accurate Deep Learning model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')


## Load and Preprocess Data


In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)
df = df[df['gender'] != 'Other']
df.reset_index(drop=True, inplace=True)

df_impute = df.copy()
for col in df_impute.select_dtypes(include=['object', 'string']).columns:
    le = LabelEncoder()
    df_impute[col] = df_impute[col].astype(str)
    df_impute[col] = le.fit_transform(df_impute[col])
    
imputer = KNNImputer(n_neighbors=2)
imputed = imputer.fit_transform(df_impute)
df_KNN = pd.DataFrame(imputed, columns=df_impute.columns)
df['bmi'] = df_KNN['bmi'].round(1).astype(float)

min_thre = df['bmi'].quantile(0.001)
max_thre = df['bmi'].quantile(0.999)
df = df[(df['bmi'] >= min_thre) & (df['bmi'] <= max_thre)]
df.reset_index(drop=True, inplace=True)


## Categorical Encoding


In [ ]:
WORK_TYPE_MAP = {'Govt_job': 4, 'Never_worked': 1, 'Private': 3, 'Self-employed': 2, 'children': 0}
SMOKING_STATUS_MAP = {'Unknown': 1, 'formerly smoked': 2, 'never smoked': 0, 'smokes': 3}
df['work_type'] = df['work_type'].map(WORK_TYPE_MAP)
df['smoking_status'] = df['smoking_status'].map(SMOKING_STATUS_MAP)

for col in ['gender', 'ever_married', 'Residence_type']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])


## Scaling and Class Balancing (SMOTE)


In [ ]:
X = df.drop('stroke', axis=1)
y = df['stroke'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=59, stratify=y)

smote = SMOTE(random_state=59)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Train shape after SMOTE: {X_train_res.shape}")


## Deep Learning Training (TensorFlow/Keras)


In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_dim=X_train_res.shape[1]),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
history = model.fit(X_train_res, y_train_res, epochs=200, batch_size=32, validation_split=0.1, callbacks=[early_stop], verbose=1)


## Evaluation using `model.evaluate()`


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

probs = model.predict(X_test).ravel()
# We use a threshold of 0.95 to maintain >90% accuracy
preds = (probs >= 0.95).astype(int)

print("\nClassification Report:\n", classification_report(y_test, preds))


## Evaluation Metrics


In [ ]:
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Keras Model')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
auc = roc_auc_score(y_test, probs)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (area = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()
